In [2]:
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv("/content/train.csv")   # adjust path if needed
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print()

# ─────────────────────────────────────────────
# Q1 — Frequency distribution of correct answers
# ─────────────────────────────────────────────
print("=" * 60)
print("Q1: Frequency Distribution of Correct Answers")
print("=" * 60)

answer_col = "answer"          # change if your column is named differently
freq = df[answer_col].value_counts().sort_index()
print(freq)

Dataset shape: (2000, 8)
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

Q1: Frequency Distribution of Correct Answers
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64


In [3]:
def clean_text(text):
    """Lowercase + remove all string.punctuation characters."""
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

df["cleaned_prompt"] = df["prompt"].apply(clean_text)

all_words = []
for text in df["cleaned_prompt"]:
    all_words.extend(text.split())

vocabulary = set(all_words)
q2_answer  = len(vocabulary)
print(f">>> Q2 Answer (unique words): {q2_answer}")
print()


>>> Q2 Answer (unique words): 859



In [4]:
print("=" * 60)
print("Q3: Words Left in Row ID 1 After Stop Word Filtering")
print("=" * 60)

# Row ID 1 — adjust index if IDs don't start at 0
row1 = df.iloc[0]
row1_words   = row1["cleaned_prompt"].split()
filtered_row1 = [w for w in row1_words if w not in ENGLISH_STOP_WORDS]

print(f"Original words : {len(row1_words)}")
print(f"After filtering: {len(filtered_row1)}")
print(f"Remaining words: {filtered_row1}")
print(f">>> Q3 Answer  : {len(filtered_row1)}")
print()

Q3: Words Left in Row ID 1 After Stop Word Filtering
Original words : 22
After filtering: 13
Remaining words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
>>> Q3 Answer  : 13



In [6]:
option_cols = ["A", "B", "C", "D", "E"]   # change if named differently

# Build combined corpus: every prompt + every option cell
corpus = []
for _, row in df.iterrows():
    corpus.append(str(row["prompt"]))
    for col in option_cols:
        corpus.append(str(row[col]))
tf = TfidfVectorizer(stop_words='english')
tf.fit(corpus)
print(len(tf.vocabulary_))

2762


In [10]:
row1_prompt = str(df.iloc[0]['prompt'])
row1_option_A = str(df.iloc[0]['A'])

vec = tf.transform([row1_prompt, row1_option_A])
print(cosine_similarity(vec[0], vec[1])[0][0])


0.2328004755964843


In [14]:
correct = 0
total   = len(df)

for _, row in df.iterrows():
    prompt_vec = tf.transform([str(row["prompt"])])
    sims = {}
    for opt in option_cols:
        opt_vec    = tf.transform([str(row[opt])])
        sims[opt]  = cosine_similarity(prompt_vec, opt_vec)[0][0]

    best_option = max(sims, key=sims.get)
    if best_option == str(row[answer_col]).strip():
        correct += 1

q6_answer = round((correct / total) * 100, 2)
print(f"Correct predictions : {correct} / {total}")
print(f">>> Q6 Answer       : {q6_answer}%")
print()


Correct predictions : 274 / 2000
>>> Q6 Answer       : 13.7%



In [15]:
def average_precision_at_k(actual, predicted, k=3):
    """
    actual    : single correct answer string, e.g. 'C'
    predicted : list of predicted answers, e.g. ['C', 'A', 'B']
    """
    predicted = predicted[:k]
    score = 0.0
    hits  = 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1)
    return score

In [17]:
q7_answer = average_precision_at_k("C", ["C", "A", "B"], k=3)
print(q7_answer)

1.0


In [18]:
q8_answer = average_precision_at_k("B", ["D", "B", "E"], k=3)
print(q8_answer)

0.5


In [19]:
top3_options = freq.sort_values(ascending=False).index[:3].tolist()
print(f"Static prediction order: {top3_options}")

q9_scores = []
for _, row in df.iterrows():
    gt    = str(row[answer_col]).strip()
    score = average_precision_at_k(gt, top3_options, k=3)
    q9_scores.append(score)

q9_answer = round(np.mean(q9_scores), 4)
print(f">>> Q9 Answer (MAP@3): {q9_answer}")
print()

Static prediction order: ['B', 'C', 'A']
>>> Q9 Answer (MAP@3): 0.4212



In [21]:
q10_scores = []

for _, row in df.iterrows():
    prompt_vec = tf.transform([str(row["prompt"])])
    sims = {}
    for opt in option_cols:
        opt_vec   = tf.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_vec, opt_vec)[0][0]

    # Sort options by similarity descending → top-3 predictions
    top3_pred = sorted(sims, key=sims.get, reverse=True)[:3]
    gt        = str(row[answer_col]).strip()
    score     = average_precision_at_k(gt, top3_pred, k=3)
    q10_scores.append(score)

q10_answer = round(np.mean(q10_scores), 4)
print(f">>> Q10 Answer (MAP@3): {q10_answer}")
print()

>>> Q10 Answer (MAP@3): 0.3119

